In [1]:
# uv pip install sqlalchemy pandas python-dotenv psycopg2-binary
from sqlalchemy import create_engine
import pandas as pd
from dotenv import load_dotenv
import os

# 환경 변수 로드
load_dotenv()
database_url = os.getenv('DATABASE_URL')
engine = create_engine(database_url)
print("db 연결 성공!")

db 연결 성공!


In [2]:
# studies 테이블에서 선택된 컬럼만 로드
query = """ 
SELECT
    nct_id,
    start_month_year,
    completion_month_year,
    study_type,
    enrollment,
    overall_status,
    phase,
    number_of_arms,  -- 그룹수
    has_dmc,         -- 안전 모니터링
    has_expanded_access,   -- 환자 접근성
    is_fda_regulated_drug, -- 약물 규제 여부
    is_fda_regulated_device  -- 기기 규제 여부
FROM ctgov.studies;
"""

df_studies = pd.read_sql(query, engine)


In [3]:
print(df_studies.head(5))
print(df_studies.shape)

        nct_id start_month_year completion_month_year      study_type  \
0  NCT00946062          2007-11               2010-07  INTERVENTIONAL   
1  NCT03964701       2019-04-22            2020-04-22   OBSERVATIONAL   
2  NCT06515847       2017-09-01            2018-09-01  INTERVENTIONAL   
3  NCT03717220       2008-07-15            2016-07-30  INTERVENTIONAL   
4  NCT03159455       2017-06-07            2017-12-16  INTERVENTIONAL   

   enrollment overall_status   phase  number_of_arms has_dmc  \
0       190.0      COMPLETED      NA             2.0   False   
1        18.0        UNKNOWN     NaN             NaN    None   
2        60.0      COMPLETED      NA             2.0   False   
3        26.0      COMPLETED      NA             1.0    None   
4        48.0      COMPLETED  PHASE1             2.0   False   

  has_expanded_access is_fda_regulated_drug is_fda_regulated_device  
0               False                  None                    None  
1               False               

In [4]:
df_studies.isnull().sum()

nct_id                          0
start_month_year             5315
completion_month_year       16702
study_type                    964
enrollment                   7084
overall_status                  0
phase                      136266
number_of_arms             160121
has_dmc                    101731
has_expanded_access          6816
is_fda_regulated_drug      223424
is_fda_regulated_device    223502
dtype: int64

In [5]:
df_studies.info()

<class 'pandas.DataFrame'>
RangeIndex: 575074 entries, 0 to 575073
Data columns (total 12 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   nct_id                   575074 non-null  str    
 1   start_month_year         569759 non-null  str    
 2   completion_month_year    558372 non-null  str    
 3   study_type               574110 non-null  str    
 4   enrollment               567990 non-null  float64
 5   overall_status           575074 non-null  str    
 6   phase                    438808 non-null  str    
 7   number_of_arms           414953 non-null  float64
 8   has_dmc                  473343 non-null  object 
 9   has_expanded_access      568258 non-null  object 
 10  is_fda_regulated_drug    351650 non-null  object 
 11  is_fda_regulated_device  351572 non-null  object 
dtypes: float64(2), object(4), str(6)
memory usage: 52.6+ MB


In [6]:
# 중재 연구 필터링, 이진 타겟 매핑
df_studies = df_studies[df_studies['study_type'] == 'INTERVENTIONAL'].copy()

final_outcomes = ['COMPLETED', 'TERMINATED', 'WITHDRAWN']

df_studies = df_studies[df_studies['overall_status'].isin(final_outcomes)].copy()

df_studies['overall_status'] = df_studies['overall_status'].map({'COMPLETED':1, 'TERMINATED':0, 'WITHDRAWN':0})


In [7]:
df_studies.info()

<class 'pandas.DataFrame'>
Index: 287815 entries, 0 to 575073
Data columns (total 12 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   nct_id                   287815 non-null  str    
 1   start_month_year         285282 non-null  str    
 2   completion_month_year    279679 non-null  str    
 3   study_type               287815 non-null  str    
 4   enrollment               284574 non-null  float64
 5   overall_status           287815 non-null  int64  
 6   phase                    287708 non-null  str    
 7   number_of_arms           267452 non-null  float64
 8   has_dmc                  239713 non-null  object 
 9   has_expanded_access      283479 non-null  object 
 10  is_fda_regulated_drug    136626 non-null  object 
 11  is_fda_regulated_device  136592 non-null  object 
dtypes: float64(2), int64(1), object(4), str(5)
memory usage: 28.5+ MB


In [8]:
print(df_studies['is_fda_regulated_drug'].unique())
print(df_studies['is_fda_regulated_device'].unique())
print(df_studies['has_dmc'].unique())
print(df_studies['has_expanded_access'].unique())


[None False True]
[None False True]
[False None True]
[False None True]


In [9]:
print(df_studies['number_of_arms'].value_counts())


number_of_arms
2.0     145967
1.0      64531
3.0      30741
4.0      15108
5.0       3972
6.0       3262
8.0        986
7.0        982
9.0        489
10.0       384
12.0       251
11.0       187
14.0       110
16.0       106
13.0       102
15.0        72
18.0        49
17.0        32
19.0        21
20.0        19
32.0        16
24.0        13
22.0        11
21.0         8
23.0         7
27.0         5
25.0         4
30.0         3
26.0         3
28.0         2
29.0         2
43.0         1
44.0         1
37.0         1
31.0         1
40.0         1
33.0         1
34.0         1
Name: count, dtype: int64


In [10]:
# 범주형 컬럼(object type) 결측치 채우기
df_studies['is_fda_regulated_drug']= df_studies['is_fda_regulated_drug'].fillna('unknown')
df_studies['is_fda_regulated_device'] = df_studies['is_fda_regulated_device'].fillna('unknown')
df_studies['has_dmc']= df_studies['has_dmc'].fillna('unknown')
df_studies['has_expanded_access'] = df_studies['has_expanded_access'].fillna('unknown')

In [11]:
# 숫자형 컬럼(float64 type) 결측치 중앙값으로 채우기
df_studies['number_of_arms'] = df_studies['number_of_arms'].fillna(df_studies['number_of_arms'].median())

In [12]:
# 날짜 컬럼(str type) 전처리
# 결측치 삭제
df_studies = df_studies.dropna(subset=['start_month_year', 'completion_month_year'])

# str -> 날짜 형식 변환
from dateutil import parser

def parse_date_safe(val):
    try:
        return parser.parse(str(val))   # 날짜 문자열을 datetime 객체로 변환
    except:
        return pd.NaT    # 변환 실패 시 "Not a Time"(빈날짜) 반환
df_studies['start_month_year'] = df_studies['start_month_year'].apply(parse_date_safe)
df_studies['completion_month_year'] = df_studies['completion_month_year'].apply(parse_date_safe)

# ⭐ 핵심 수정: 판다스가 날짜 컬럼임을 확실히 인식하도록 변환 (AttributeError 방지)
df_studies['start_month_year'] = pd.to_datetime(df_studies['start_month_year'], errors='coerce')
df_studies['completion_month_year'] = pd.to_datetime(df_studies['completion_month_year'], errors='coerce')

# 연구 기간 계산 
df_studies['duration_of_study'] = (df_studies['completion_month_year'] - df_studies['start_month_year']).dt.days


In [13]:
df_studies['duration_of_study'].value_counts()

duration_of_study
365     3439
61      2133
730     1900
92      1842
153     1840
        ... 
5066       1
3141       1
4166       1
5879       1
4878       1
Name: count, Length: 5329, dtype: int64

In [14]:
df_studies['phase'].unique()


<StringArray>
[           'NA',        'PHASE1',        'PHASE4',        'PHASE2',
        'PHASE3',  'EARLY_PHASE1', 'PHASE1/PHASE2', 'PHASE2/PHASE3',
             nan]
Length: 9, dtype: str

In [15]:
df_studies['phase'].value_counts()

phase
NA               133708
PHASE2            40733
PHASE1            34649
PHASE3            28530
PHASE4            23786
PHASE1/PHASE2      9541
PHASE2/PHASE3      4526
EARLY_PHASE1       3225
Name: count, dtype: int64

In [16]:
# phase 컬럼 (str type) 결측치 전처리
df_studies['phase'] = df_studies['phase'].replace({'NA':'unknown', 'None':'unknown'})
df_studies['phase'] = df_studies['phase'].fillna('unknown')
# 소문자 변환
df_studies['phase'] = df_studies['phase'].str.lower()

In [17]:
df_studies['phase'].value_counts()

phase
unknown          133815
phase2            40733
phase1            34649
phase3            28530
phase4            23786
phase1/phase2      9541
phase2/phase3      4526
early_phase1       3225
Name: count, dtype: int64

In [18]:
# enrollment 컬럼 (flaot64 type) 전처리
# 결측치 확인
df_studies['enrollment'].unique()

array([  190.,    60.,    26., ...,  4325., 16264., 14250.], shape=(5036,))

In [19]:
df_studies['enrollment'].value_counts()

enrollment
0.0        12185
30.0        7537
60.0        7521
20.0        7209
40.0        7164
           ...  
4744.0         1
8637.0         1
4325.0         1
16264.0        1
14250.0        1
Name: count, Length: 5035, dtype: int64

In [20]:
df_studies['enrollment'].isnull().sum()

np.int64(1893)

In [21]:
# 결측치 채우기 - 임상 단계별 환자수
# 임상 1상과 3상 참여 환자수 규모의 갭이 매우 큼. 따라서 각 그룹별 중앙값 계산해서 채움
df_studies['enrollment'] = df_studies.groupby('phase')['enrollment'].transform(lambda x: x.fillna(x.median()))

df_studies['enrollment'].isnull().sum()


np.int64(0)

In [22]:
df_studies['study_type'].value_counts()

study_type
INTERVENTIONAL    278805
Name: count, dtype: int64

In [23]:
df_studies['study_type'].isnull().sum()

np.int64(0)

In [24]:
df_studies['overall_status'].value_counts()

overall_status
1    238725
0     40080
Name: count, dtype: int64

In [25]:
df_studies['overall_status'].isnull().sum()

np.int64(0)

In [26]:
# 모호한 phase 단계 재분류
# phase1/2 : (50명 기준, 소규모는 phase1, 대규모는 phase2)
# phase2/3 : (2상과 3상 중간값의 평균 기준. 기준보다 작으면 phase2, 크면 phase3)
# early_phase1 :  분석의 단순화를 위해 phase1과 함침

size = 50
df_studies.loc[(df_studies['phase'] == 'phase1/phase2') & (df_studies['enrollment'] <= size), 'phase'] = 'phase1'
df_studies.loc[(df_studies['phase'] == 'phase1/phase2') & (df_studies['enrollment'] > size), 'phase'] = 'phase1'

median_size = (df_studies[df_studies['phase'] == 'phase2']['enrollment'].median() + df_studies[df_studies['phase'] == 'phase3']['enrollment'].median())/2
df_studies.loc[(df_studies['phase'] == 'phase2/phase3') & (df_studies['enrollment'] <= median_size), 'phase'] = 'phase2'
df_studies.loc[(df_studies['phase'] == 'phase2/phase3') & (df_studies['enrollment'] > median_size), 'phase'] = 'phase3'

df_studies['phase'] = df_studies['phase'].replace({'early_phase1':'phase1'})

df_studies['phase'].value_counts()

phase
unknown    133815
phase1      47415
phase2      43730
phase3      30059
phase4      23786
Name: count, dtype: int64

In [27]:
# 불리언 데이터(true/false)를 숫자 형태로 변환 (binary encoding)
# 머신러닝 모델이 데이터 읽을 수 있도록 (1-긍정, 0-부정, -1:알수없음)

binary_columns = ['has_dmc', 'is_fda_regulated_device', 'is_fda_regulated_drug', 'has_expanded_access']
for col in binary_columns:
    df_studies[col] = df_studies[col].map({True:1, False:0, 'unknown':-1})

In [28]:
# 범주형 데이터(phase1,2,3,4) 원핫 인코딩 
# phase 컬럼 단순화 (데이터깔끔하게)
df_studies['phase']= df_studies['phase'].str.replace('phase', '', regex=False)

# 원핫인코딩 파생변수 생성
df_studies = pd.get_dummies(df_studies, columns=['phase'], prefix='phase', dtype=int)

df_studies.head()

,nct_id,start_month_year,completion_month_year,study_type,enrollment,overall_status,number_of_arms,has_dmc,has_expanded_access,is_fda_regulated_drug,is_fda_regulated_device,duration_of_study,phase_1,phase_2,phase_3,phase_4,phase_unknown
0,NCT00946062,2007-11-25,2010-07-25,INTERVENTIONAL,190.0,1,2.0,0,0,-1,-1,973,0,0,0,0,1
2,NCT06515847,2017-09-01,2018-09-01,INTERVENTIONAL,60.0,1,2.0,0,0,0,0,365,0,0,0,0,1
3,NCT03717220,2008-07-15,2016-07-30,INTERVENTIONAL,26.0,1,1.0,-1,0,0,0,2937,0,0,0,0,1
4,NCT03159455,2017-06-07,2017-12-16,INTERVENTIONAL,48.0,1,2.0,0,0,0,0,192,1,0,0,0,0
5,NCT00766675,2008-10-25,2009-04-25,INTERVENTIONAL,80.0,1,1.0,0,0,-1,-1,182,0,0,0,1,0


In [29]:
df_studies.head()

,nct_id,start_month_year,completion_month_year,study_type,enrollment,overall_status,number_of_arms,has_dmc,has_expanded_access,is_fda_regulated_drug,is_fda_regulated_device,duration_of_study,phase_1,phase_2,phase_3,phase_4,phase_unknown
0,NCT00946062,2007-11-25,2010-07-25,INTERVENTIONAL,190.0,1,2.0,0,0,-1,-1,973,0,0,0,0,1
2,NCT06515847,2017-09-01,2018-09-01,INTERVENTIONAL,60.0,1,2.0,0,0,0,0,365,0,0,0,0,1
3,NCT03717220,2008-07-15,2016-07-30,INTERVENTIONAL,26.0,1,1.0,-1,0,0,0,2937,0,0,0,0,1
4,NCT03159455,2017-06-07,2017-12-16,INTERVENTIONAL,48.0,1,2.0,0,0,0,0,192,1,0,0,0,0
5,NCT00766675,2008-10-25,2009-04-25,INTERVENTIONAL,80.0,1,1.0,0,0,-1,-1,182,0,0,0,1,0


In [30]:
# 필요없는 컬럼 삭제
df_studies = df_studies.drop(columns=['start_month_year', 'completion_month_year'])



In [31]:
df_studies.info()

<class 'pandas.DataFrame'>
Index: 278805 entries, 0 to 575073
Data columns (total 15 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   nct_id                   278805 non-null  str    
 1   study_type               278805 non-null  str    
 2   enrollment               278805 non-null  float64
 3   overall_status           278805 non-null  int64  
 4   number_of_arms           278805 non-null  float64
 5   has_dmc                  278805 non-null  int64  
 6   has_expanded_access      278805 non-null  int64  
 7   is_fda_regulated_drug    278805 non-null  int64  
 8   is_fda_regulated_device  278805 non-null  int64  
 9   duration_of_study        278805 non-null  int64  
 10  phase_1                  278805 non-null  int64  
 11  phase_2                  278805 non-null  int64  
 12  phase_3                  278805 non-null  int64  
 13  phase_4                  278805 non-null  int64  
 14  phase_unknown       

In [32]:
# 파일 저장

target_path = r'C:\dev\clinicaltrials-study\data\processed'

if not os.path.exists(target_path):
    os.makedirs(target_path, exist_ok=True)
    print(f"폴더 생성 완료:{target_path}")

file_name = 'studies_clean.csv'
full_save_path = os.path.join(target_path, file_name)
df_studies.to_csv(full_save_path, index=False)


In [33]:
# 확인하고 싶은 파일의 전체 경로
check_path = r'C:\dev\clinicaltrials-study\data\processed\studies_clean.csv'

if os.path.exists(check_path):
    # 파일 정보 가져오기
    file_stats = os.stat(check_path)
    print(f"✅ 파일이 존재합니다: {check_path}")
    print(f"📁 파일 크기: {file_stats.st_size / 1024:.2f} KB") # 0KB라면 데이터가 안 담긴 것
    print(f"🕒 마지막 수정 시간: {os.path.getmtime(check_path)}")
else:
    print(f"❌ 파일이 없습니다. 경로를 다시 확인하세요: {check_path}")
    
    # 현재 파이썬이 인식하는 '작업 디렉토리' 확인 (상대 경로 문제 해결용)
    print(f"📍 현재 작업 디렉토리: {os.getcwd()}")

✅ 파일이 존재합니다: C:\dev\clinicaltrials-study\data\processed\studies_clean.csv
📁 파일 크기: 17067.11 KB
🕒 마지막 수정 시간: 1774409054.8629332
